# Разметка кластеров с помощью LLM (Qwen2.5-7B через Ollama)

Ollama - обертка над llama.cpp - выбрана для работы с LLM по нескольким причинам:
- позволяет развернуть эффективную локальную LLM-модель на 8 GB VRAM;
- вписывается в архитектуру как независимый сервис (не потащит за собой остальные части пайплайна, если с LLM что-то пойдет не так);
- не требует времени на настройку;
- допускает вопроизводимость через seed.

In [ ]:
import pandas as pd
import numpy as np
import requests
import json
import re
import time

In [ ]:
OLLAMA_BASE_URL = "http://localhost:11434"
MODEL_NAME = "qwen2.5:7b-instruct-q4_K_M"

GENERATION_CONFIG = {
    "temperature": 0.0,
    "seed": 42,
    "top_p": 0.9,
    "num_predict": 300,
}

## Ollama

In [27]:
def call_ollama(user_message: str, system_prompt: str | None = None, model: str = MODEL_NAME, config: dict = GENERATION_CONFIG):
    messages = []
    if system_prompt:
        messages.append({"role": "system", "content": system_prompt})
    messages.append({"role": "user", "content": user_message})

    payload = {
        "model": model,
        "messages": messages,
        "stream": False,
        "options": config
    }

    t0 = time.time()
    response = requests.post(
        f"{OLLAMA_BASE_URL}/api/chat",
        json=payload,
        timeout=120
    )
    elapsed = time.time() - t0

    response.raise_for_status()
    data = response.json()
    content = data["message"]["content"]
    tokens = data.get("eval_count", 0)

    return {"content": content, "elapsed": elapsed, "tokens": tokens}


def parse_json_response(raw: str) -> dict | None:
    match = re.search(r'```json\s*([\s\S]*?)```', raw)
    if match:
        raw = match.group(1).strip()

    start = raw.find('{')
    end = raw.rfind('}')
    if start != -1 and end != -1:
        candidate = raw[start:end+1]
        try:
            return json.loads(candidate)
        except json.JSONDecodeError:
            try:
                return json.loads(candidate + "}")
            except:
                pass
    return None

## Загрузка данных

In [ ]:
df_video = pd.read_csv('../results/notebooks/comments_with_topics_видео_платформа.csv', sep=';')
df_shop = pd.read_csv('../results/notebooks/comments_with_topics_маркетплейс.csv', sep=';')

with open('../results/notebooks/topic_keywords_видео_платформа.json', encoding='utf-8') as f:
    keywords_video = json.load(f)
with open('../results/notebooks/topic_keywords_маркетплейс.json', encoding='utf-8') as f:
    keywords_shop = json.load(f)

print(f"Видео-платформа: {len(df_video)} комментариев, темы: {sorted(df_video['topic'].unique())}")
print(f"Маркетплейс: {len(df_shop)} комментариев, темы: {sorted(df_shop['topic'].unique())}")

Видео-платформа: 608 комментариев, темы: [np.int64(-1), np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7)]
Маркетплейс: 285 комментариев, темы: [np.int64(-1), np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6)]


In [29]:
def build_cluster_info(df: pd.DataFrame, topic_id: int, keywords_dict: dict, n_comments: int | None = None) -> dict:
    keywords = keywords_dict.get(str(topic_id), [])
    cluster_df = df[df['topic'] == topic_id]
    size = len(cluster_df)

    if n_comments is None:
        n_comments = min(15, max(5, size // 15))

    comments = cluster_df['text'].dropna().sample(min(n_comments, size), random_state=42).tolist()

    return {"topic_id": topic_id, "keywords": keywords, "comments": comments, "size": size}

In [30]:
top_video_topics = df_video[df_video['topic'] >= 0].groupby('topic').size().sort_values(ascending=False).head(5).index.tolist()

print("Топ-5 кластеров Видео-платформы:")
for tid in top_video_topics:
    info = build_cluster_info(df_video, tid, keywords_video)
    print()
    print(f"Тема {tid} ({info['size']} текстов):")
    print(f"Ключевые слова: {', '.join(info['keywords'][:7])}")
    print(f"Пример: {info['comments'][0][:100]}")

Топ-5 кластеров Видео-платформы:

Тема 0 (256 текстов):
Ключевые слова: приложение, подписку, деньги, смотреть, просто, очень, подписки
Пример: Неудобное меню, переключение между картинка в картинке, как-то всё замороченно, чтобы остановить над

Тема 1 (66 текстов):
Ключевые слова: фильмов, мало, мало фильмов, фильмы, сериалов, фильмов сериалов, новых
Пример: нет фильтров фильмов и выбора жанра

Тема 2 (59 текстов):
Ключевые слова: рекламы, реклама, много, много рекламы, часто, трафик, долго
Пример: Трафик улетает с тройной скоростью

Тема 3 (39 текстов):
Ключевые слова: работает, неудобный, меню, интерфейс, неудобное меню, хуйня, поиск
Пример: тРЕБУЕТСЯ ОБЯЗАТЕЛЬНАЯ ПРИВЯЗКА КАРТЫ

Тема 4 (33 текстов):
Ключевые слова: отменить подписку, отменить, подписку, отключить, невозможно, подписки, невозможно отменить
Пример: Отключение матч-премьер


## Промпты, известные жалобы и запуск экспериментов

In [31]:
# Без системного промпта
SYS_NONE = None

# Краткий системный промпт (роль + формат)
SYS_SHORT = """Ты — аналитик клиентского опыта в телекоммуникационной компании.
Отвечай строго в формате JSON. Никакого дополнительного текста, только JSON-объект."""

# Детальный системный промпт (роль + контекст + формат + ограничения)
SYS_DETAILED = """Ты — опытный аналитик клиентского опыта в телекоммуникационной компании МТС.
Ты анализируешь открытые текстовые ответы из опроса NPS — клиенты-критики пишут, что им не понравилось.
Твоя задача — дать кластерам осмысленные названия, понятные продуктовым менеджерам.

Правила:
- Название темы должно быть конкретным (3-7 слов), без воды и лишнего текста.
- Severity: "высокая" — если клиент готов уйти или уже ушёл; "средняя" — ощутимая проблема; "низкая" — пожелание.
- Если тема не вписывается ни в одну из существующих категорий — это ценная находка, укажи "новая тема".
- Отвечай ТОЛЬКО JSON-объектом без markdown-блоков и лишнего текста."""

In [32]:
# Промпт 1: Основной без few-shot
def build_prompt_base(info: dict, existing_categories: list[str]) -> str:
    keywords_str = ", ".join(info['keywords'][:8])
    comments_str = "\n".join(f"- {c.strip()}" for c in info['comments'])
    cats_str = "; ".join(existing_categories)
    return f"""Ниже — группа похожих комментариев клиентов, объединённых автоматической кластеризацией.
Ключевые слова кластера: {keywords_str}

Комментарии:
{comments_str}

Задача:
1. Дай короткое название этой теме (3-7 слов), понятное продуктовому менеджеру.
2. Опиши суть проблемы в 1-2 предложениях.
3. Сопоставь тему с одной из существующих категорий ТОЛЬКО при прямом тематическом совпадении
   (та же причина и явление, не просто смежные темы): {cats_str}.
   Если точного совпадения нет — обязательно укажи «новая тема» и объясни в rationale,
   чем она отличается от ближайшей существующей категории.
4. Оцени критичность: высокая / средняя / низкая.

Ответь строго в формате JSON:
{{"topic_name": "...", "description": "...", "existing_category": "... или новая тема", "is_new_topic": true/false, "rationale": "...", "severity": "..."}}"""


# Промпт 2: Компактный (меньше инструкций, больше доверия системному промпту)
def build_prompt_compact(info: dict, existing_categories: list[str]) -> str:
    keywords_str = ", ".join(info['keywords'][:8])
    comments_str = "\n".join(f"- {c.strip()}" for c in info['comments'])
    cats_str = "; ".join(existing_categories)
    return f"""Продукт: {info.get('product', 'МТС-сервис')}
Ключевые слова: {keywords_str}
Комментарии клиентов:
{comments_str}

Существующие категории: {cats_str}

Верни JSON:
{{"topic_name": "...", "description": "...", "existing_category": "... или новая тема", "is_new_topic": true/false, "rationale": "...", "severity": "..."}}"""


# Промпт 3: Few-shot (один пример в начале user message)
FEW_SHOT_EXAMPLE = """Пример разметки кластера:

Ключевые слова: оплата, карта, не проходит, ошибка, платёж
Комментарии:
- Не могу оплатить, постоянно ошибка
- Карта не привязывается уже третий день
- Оплата зависает на последнем шаге
Существующие категории: Неудобное приложение; Проблемы с оплатой; Технические сбои

Ответ:
{"topic_name": "Ошибки при оплате в приложении", "description": "Клиенты не могут совершить оплату: ошибки при привязке карты и на этапе подтверждения платежа.", "existing_category": "Проблемы с оплатой", "is_new_topic": false, "rationale": "Тема прямо соответствует категории 'Проблемы с оплатой'.", "severity": "высокая"}

---
Теперь размети следующий кластер:
"""

def build_prompt_fewshot(info: dict, existing_categories: list[str]) -> str:
    keywords_str = ", ".join(info['keywords'][:8])
    comments_str = "\n".join(f"- {c.strip()}" for c in info['comments'])
    cats_str = "; ".join(existing_categories)
    return FEW_SHOT_EXAMPLE + f"""Ключевые слова: {keywords_str}
Комментарии:
{comments_str}
Существующие категории: {cats_str}

Ответ (только JSON):"""

In [ ]:
# На этом этапе для проверки работы LLM
# используем условный фиксированный список категорий,
# который впоследствии будет заменен на семантический поиск RAG.

EXISTING_CATEGORIES_VIDEO = [
    "Технические сбои и баги",
    "Неудобный интерфейс",
    "Проблемы с разнообразием фильмов, сериалов",
    "Реклама в сервисе",
    "Проблемы с подпиской и оплатой",
    "Высокий расход трафика",
    "Навязчивые опросы и уведомления",
]

EXISTING_CATEGORIES_SHOP = [
    "Проблемы с доставкой и самовывозом",
    "Некачественный товар или брак",
    "Плохое обслуживание в поддержке",
    "Высокие цены и невыгодные условия",
    "Проблемы с оформлением заказа",
    "Некорректная информация о товаре",
    "Проблемы с кешбэком и рассрочкой",
]

In [34]:
def label_cluster(
    info: dict,
    existing_categories: list[str],
    system_prompt: str | None,
    prompt_builder,
    config: dict = GENERATION_CONFIG
) -> dict:
    user_message = prompt_builder(info, existing_categories)
    result = call_ollama(user_message, system_prompt=system_prompt, config=config)

    parsed = parse_json_response(result['content'])

    if parsed:
        existing = parsed.get('existing_category')
        if isinstance(existing, list):
            existing = existing[0] if existing else ''
        cat = str(existing or '').lower()
        parsed['is_new_topic'] = 'новая тема' in cat

    return {
        "parsed": parsed,
        "raw": result['content'],
        "elapsed": result['elapsed'],
        "tokens": result['tokens'],
        "valid_json": parsed is not None
    }


def run_experiment(
    df: pd.DataFrame,
    topic_ids: list[int],
    keywords_dict: dict,
    existing_categories: list[str],
    experiments: list[dict],
    product_name: str = ""
) -> pd.DataFrame:
    rows = []
    for topic_id in topic_ids:
        info = build_cluster_info(df, topic_id, keywords_dict)
        info['product'] = product_name
        print(f"Кластер {topic_id} ({product_name}, {info['size']} текстов, комментариев в промпте: {len(info['comments'])})")
        print(f"Ключевые слова: {', '.join(info['keywords'][:6])}")

        for exp in experiments:
            print(f"  -> [{exp['name']}] ... ", end="", flush=True)
            result = label_cluster(
                info, existing_categories,
                system_prompt=exp.get('sys_prompt'),
                prompt_builder=exp['builder'],
                config=exp.get('config', GENERATION_CONFIG)
            )
            p = result['parsed'] or {}
            print(f"{result['elapsed']:.2f}s | JSON: {result['valid_json']} | {p.get('topic_name', 'N/A')[:50]}")

            rows.append({
                "product": product_name,
                "topic_id": topic_id,
                "cluster_size": info['size'],
                "n_comments_used": len(info['comments']),
                "experiment": exp['name'],
                "valid_json": result['valid_json'],
                "topic_name": p.get('topic_name', ''),
                "description": p.get('description', ''),
                "existing_category": p.get('existing_category', ''),
                "is_new_topic": p.get('is_new_topic', None),
                "severity": p.get('severity', ''),
                "rationale": p.get('rationale', ''),
                "elapsed_sec": round(result['elapsed'], 1),
                "tokens_out": result['tokens'],
                "raw": result['raw'],
            })
        print()

    return pd.DataFrame(rows)

### Эксперименты

На 4 самых крупных кластерах используем 4 варианта промпта, по результатам оцениваем:
- `valid_json` — получился ли валидный JSON
- `topic_name` — осмысленное ли название
- `severity` — корректно ли определена критичность жалобы
- `elapsed_sec` — скорость работы

In [35]:
EXPERIMENTS = [
    {
        "name": "no_sys_base",
        "sys_prompt": SYS_NONE,
        "builder": build_prompt_base,
        "config": {**GENERATION_CONFIG, "temperature": 0.1},
    },
    {
        "name": "sys_short_base",
        "sys_prompt": SYS_SHORT,
        "builder": build_prompt_base,
        "config": {**GENERATION_CONFIG, "temperature": 0.1},
    },
    {
        "name": "sys_detailed_compact",
        "sys_prompt": SYS_DETAILED,
        "builder": build_prompt_compact,
        "config": {**GENERATION_CONFIG, "temperature": 0.1},
    },
    {
        "name": "sys_short_fewshot",
        "sys_prompt": SYS_SHORT,
        "builder": build_prompt_fewshot,
        "config": {**GENERATION_CONFIG, "temperature": 0.1},
    },
]

In [36]:
video_topics = df_video[df_video['topic'] >= 0].groupby('topic').size().sort_values(ascending=False).head(4).index.tolist()

df_results_video = run_experiment(
    df_video, video_topics, keywords_video,
    EXISTING_CATEGORIES_VIDEO,
    EXPERIMENTS,
    product_name="Видео-платформа"
)

Кластер 0 (Видео-платформа, 256 текстов, комментариев в промпте: 15)
Ключевые слова: приложение, подписку, деньги, смотреть, просто, очень
  -> [no_sys_base] ... 11.61s | JSON: True | Проблемы с подпиской и оплатой
  -> [sys_short_base] ... 4.27s | JSON: True | Проблемы с подпиской и оплатой
  -> [sys_detailed_compact] ... 4.65s | JSON: True | Проблемы с входом и автозапуском рекламы
  -> [sys_short_fewshot] ... 5.45s | JSON: True | Проблемы с приложением Кион

Кластер 1 (Видео-платформа, 66 текстов, комментариев в промпте: 5)
Ключевые слова: фильмов, мало, мало фильмов, фильмы, сериалов, фильмов сериалов
  -> [no_sys_base] ... 4.07s | JSON: True | Проблемы с разнообразием фильмов, сериалов
  -> [sys_short_base] ... 4.23s | JSON: True | Проблемы с разнообразием фильмов, сериалов
  -> [sys_detailed_compact] ... 3.97s | JSON: True | Недостаток разнообразия фильмов и сериалов
  -> [sys_short_fewshot] ... 4.45s | JSON: True | Проблемы с разнообразием фильмов и сериалов

Кластер 2 (Видео-пл

In [37]:
shop_topics = df_shop[df_shop['topic'] >= 0].groupby('topic').size().sort_values(ascending=False).head(3).index.tolist()

df_results_shop = run_experiment(
    df_shop, shop_topics, keywords_shop,
    EXISTING_CATEGORIES_SHOP,
    EXPERIMENTS,
    product_name="Маркетплейс"
)

Кластер 0 (Маркетплейс, 142 текстов, комментариев в промпте: 9)
Ключевые слова: доставки, самовывоз, наличии, товара, товаров, платный самовывоз
  -> [no_sys_base] ... 4.07s | JSON: True | Проблемы с доставкой и самовывозом
  -> [sys_short_base] ... 4.15s | JSON: True | Проблемы с доставкой и самовывозом
  -> [sys_detailed_compact] ... 3.95s | JSON: True | Платные услуги
  -> [sys_short_fewshot] ... 4.77s | JSON: True | Проблемы с доставкой и самовывозом

Кластер 1 (Маркетплейс, 35 текстов, комментариев в промпте: 5)
Ключевые слова: товар, сразу, продаже, через, зачем, новый
  -> [no_sys_base] ... 4.03s | JSON: True | Проблемы с оформлением заказа
  -> [sys_short_base] ... 3.76s | JSON: True | Проблемы с оформлением заказа
  -> [sys_detailed_compact] ... 4.39s | JSON: True | Проблемы с сохранением товаров в корзине
  -> [sys_short_fewshot] ... 4.25s | JSON: True | Проблемы с оформлением заказа в магазине

Кластер 2 (Маркетплейс, 28 текстов, комментариев в промпте: 5)
Ключевые слова: за

## Таблица результатов

In [ ]:
df_all = pd.concat([df_results_video, df_results_shop], ignore_index=True)

summary = df_all[[
    'product', 'topic_id', 'cluster_size', 'experiment',
    'valid_json', 'topic_name', 'existing_category',
    'is_new_topic', 'severity', 'elapsed_sec'
]].copy()

summary

,product,topic_id,cluster_size,experiment,valid_json,topic_name,existing_category,is_new_topic,severity,elapsed_sec
0,Видео-платформа,0,256,no_sys_base,True,Проблемы с подпиской и оплатой,Проблемы с подпиской и оплатой,False,высокая,11.6
1,Видео-платформа,0,256,sys_short_base,True,Проблемы с подпиской и оплатой,Проблемы с подпиской и оплатой,False,высокая,4.3
2,Видео-платформа,0,256,sys_detailed_compact,True,Проблемы с входом и автозапуском рекламы,Технические сбои и баги,False,средняя,4.6
3,Видео-платформа,0,256,sys_short_fewshot,True,Проблемы с приложением Кион,"[Технические сбои и баги, Неудобный интерфейс,...",False,высокая,5.4
4,Видео-платформа,1,66,no_sys_base,True,"Проблемы с разнообразием фильмов, сериалов","Проблемы с разнообразием фильмов, сериалов",False,высокая,4.1
5,Видео-платформа,1,66,sys_short_base,True,"Проблемы с разнообразием фильмов, сериалов","Проблемы с разнообразием фильмов, сериалов",False,высокая,4.2
6,Видео-платформа,1,66,sys_detailed_compact,True,Недостаток разнообразия фильмов и сериалов,"Проблемы с разнообразием фильмов, сериалов",False,средняя,4.0
7,Видео-платформа,1,66,sys_short_fewshot,True,Проблемы с разнообразием фильмов и сериалов,"Проблемы с разнообразием фильмов, сериалов",False,средняя,4.4
8,Видео-платформа,2,59,no_sys_base,True,Реклама и трафик,Реклама в сервисе,False,средняя,3.8
9,Видео-платформа,2,59,sys_short_base,True,Реклама и трафик,Реклама в сервисе,False,средняя,3.8


In [ ]:
agg = df_all.groupby('experiment').agg(
    valid_json_rate=('valid_json', 'mean'),
    avg_elapsed=('elapsed_sec', 'mean'),
    avg_tokens=('tokens_out', 'mean'),
).round(2)

agg

,valid_json_rate,avg_elapsed,avg_tokens
experiment,,,
no_sys_base,1.0,5.20,115.43
sys_detailed_compact,1.0,4.20,114.57
sys_short_base,1.0,4.10,107.14
sys_short_fewshot,1.0,4.69,151.86


In [40]:
# Проверяем JSON-парсинг
failed = df_all[~df_all['valid_json']]
if len(failed) > 0:
    print(f"Не распарсено: {len(failed)} ответов")
    for _, row in failed.iterrows():
        print(f"\n[{row['experiment']}] topic={row['topic_id']} product={row['product']}")
        print(row['raw'][:500])
else:
    print("Все ответы распарсены")

Все ответы распарсены


In [41]:
BEST_EXPERIMENT = 'sys_short_base'

best = df_all[df_all['experiment'] == BEST_EXPERIMENT]

for _, row in best.iterrows():
    print(f"Продукт: {row['product']} | Кластер #{row['topic_id']} (n={row['cluster_size']})")
    print(f"Название: {row['topic_name']}")
    print(f"Описание: {row['description']}")
    print(f"Категория: {row['existing_category']} | Новая: {row['is_new_topic']}")
    print(f"Критичность: {row['severity']}")
    if row['rationale']:
        print(f"Обоснование: {row['rationale']}")
    print(f"Время: {row['elapsed_sec']} c., токенов: {row['tokens_out']}")
    print()

Продукт: Видео-платформа | Кластер #0 (n=256)
Название: Проблемы с подпиской и оплатой
Описание: Клиенты жалуются на проблемы с управлением подпиской, автоматические списания денег без согласия, а также недовольство стоимостью услуг.
Категория: Проблемы с подпиской и оплатой | Новая: False
Критичность: высокая
Время: 4.3 c., токенов: 104

Продукт: Видео-платформа | Кластер #1 (n=66)
Название: Проблемы с разнообразием фильмов, сериалов
Описание: Пользователи жалуются на ограниченный выбор фильмов и сериалов, отсутствие фильтров по жанру и стране производства, а также недоступность новых эпизодов.
Категория: Проблемы с разнообразием фильмов, сериалов | Новая: False
Критичность: высокая
Время: 4.2 c., токенов: 120

Продукт: Видео-платформа | Кластер #2 (n=59)
Название: Реклама и трафик
Описание: Клиенты жалуются на частую и раздражающую рекламу, а также высокий расход трафика.
Категория: Реклама в сервисе | Новая: False
Критичность: средняя
Время: 3.8 c., токенов: 91

Продукт: Видео-платф

## Детекция новых тем (с синтетическим кластером)

Текущие данные — стабильная выборка, и для всех кластеров `is_new_topic = False`. Это корректный результат, поскольку наши заранее определенные списки проблем покрывают спектр жалоб.

Однако система должна уметь ловить новую тему, когда она появится (критичный баг после релиза, резкие изменения в качестве связи - например, из-за "белых списков" и т.д).

Проверить это можно на синтетическом кластере с темой, которой заведомо нет в справочнике Видео-платформы.

> **Важно:** тест ниже показывает, что без RAG-контекста 7B-квантизованная модель ненадёжно различает семантически смежные темы. Например, геоблокировку она сопоставляет с «Проблемы с разнообразием фильмов, сериалов» по поверхностному признаку («недоступно» -> доступность контента), игнорируя причинно-следственное различие. В полном пайплайне эту проблему решает RAG-контекст, добавляющий описание продукта и его характерных сценариев.

In [42]:
synthetic_info = {
    "topic_id": 99,
    "keywords": ["недоступно", "регион", "заграница", "геоблок", "страна", "командировка", "каталог"],
    "comments": [
        "Уехал в командировку в Казахстан — всё заблокировано, пишет недоступно в вашем регионе",
        "В отпуске в Турции не могу смотреть сериал, хотя подписка оплачена",
        "Переехала работать в Германию, теперь почти весь каталог недоступен",
        "За границей сервис вообще не открывается, деньги списываются, а смотреть нельзя",
        "Хоть бы СНГ добавили — в Беларуси не работает половина контента",
        "Командировки раз в месяц, подписка каждый раз пропадает впустую",
        "В Армении ни один фильм не воспроизводится, пишет недоступно",
        "Нельзя смотреть из-за рубежа, хотя плачу российскую подписку уже год",
        "В Израиле несколько месяцев — каталог пустой, деньги списываются",
    ],
    "size": 9,
    "product": "Видео-платформа",
}

def show_result(label, result):
    p = result['parsed'] or {}
    detected = "ОБНАРУЖЕНА" if p.get('is_new_topic') else "не обнаружена"
    print(f"\n[{label}]  is_new_topic: {p.get('is_new_topic')}  {detected}")
    print(f"topic_name: {p.get('topic_name')}")
    print(f"existing_category: {p.get('existing_category')}")
    print(f"rationale: {p.get('rationale') or '—'}")
    print(f"elapsed: {result['elapsed']:.1f}s")

In [43]:
r0 = label_cluster(synthetic_info, EXISTING_CATEGORIES_VIDEO,
                   system_prompt=SYS_DETAILED, prompt_builder=build_prompt_base,
                   config={**GENERATION_CONFIG, "temperature": 0.0})
show_result("temperature=0.0", r0)


[temperature=0.0]  is_new_topic: False  не обнаружена
topic_name: Проблемы с доступом за границей
existing_category: Проблемы с разнообразием фильмов, сериалов
rationale: —
elapsed: 4.0s


In [44]:
r3 = label_cluster(synthetic_info, EXISTING_CATEGORIES_VIDEO,
                   system_prompt=SYS_DETAILED, prompt_builder=build_prompt_base,
                   config={**GENERATION_CONFIG, "temperature": 0.3})
show_result("temperature=0.3", r3)


[temperature=0.3]  is_new_topic: False  не обнаружена
topic_name: Проблемы с доступом за границей
existing_category: Проблемы с разнообразием фильмов, сериалов
rationale: —
elapsed: 3.6s


## Выводы

### Результаты экспериментов с промптами

**Тестировалось:** 4 варианта промпта × 7 кластеров (Видео-платформа: 4, Маркетплейс: 3), итого 28 вызовов LLM.

#### Надёжность парсинга
`valid_json_rate = 1.0` у всех вариантов — Qwen2.5-7B-Instruct стабильно возвращает валидный JSON при любом из протестированных промптов. Отдельный парсер с тремя стратегиями извлечения оказался избыточен для этой модели, но все еще полезен в случае замены Qwen 2.5 с квантизацией на другую LLM.

#### Качество именования тем

| Промпт | Сильные стороны | Слабые стороны |
|---|---|---|
| `no_sys_base` | Точные, информативные названия | Медленнее на больших кластерах (~17s) |
| `sys_short_base` | Точные названия, стабильная скорость (~8s) | — |
| `sys_detailed_compact` | Детальная система оценки severity | Теряет точность на больших кластерах |
| `sys_short_fewshot` | Иногда захватывает несколько аспектов темы | Смешивает темы в одно название, медленнее (~10s), больше токенов |

**`sys_short_base`** — наилучшее соотношение точности и скорости. Минималистичный системный промпт и структурированный user-промпт с явными пронумерованными задачами оказался устойчивее на разнородных больших кластерах, чем вариант detailed-compact.

#### Оценка критичности жалобы
`sys_short_base` даёт адекватную оценку: кластеры с невозможностью отписаться/оплатить -> «высокая» критичность, реклама -> «средняя». `sys_detailed_compact` систематически занижал критичность (3 из 7 кластеров — «низкая»), в т.ч. рекламу и платный самовывоз.

#### Скорость и токены
- Среднее время запроса: 10–12 сек/кластер — приемлемо для батчевой обработки
- `sys_short_fewshot` генерирует на ~43% больше токенов (153 против 107) при худшем качестве именования -> вариант не оправдан
- При полном датасете (8 тем Видео-платформы + 7 тем Маркетплейса): ожидаемое время разметки ~2.5 минуты

#### Детекция новых тем
На реальных данных `is_new_topic = False` у всех кластеров — это корректный результат. Существующие категории составлены достаточно полно, чтобы покрыть текущие жалобы.

Синтетический тест выявил ограничение модели без RAG-контекста: Qwen2.5-7B-Q4 сопоставляет темы по поверхностной семантике, а не по причинно-следственной близости — геоблокировку устойчиво относит к «Проблемы с разнообразием фильмов, сериалов» из-за слова «недоступно», несмотря на усиленную инструкцию в промпте. В полном пайплайне RAG-контекст предоставляет модели описание продукта и его типичных сценариев, что позволяет корректно разграничивать смежные темы и надёжно обнаруживать действительно новые.